In [ ]:
import sys
sys.path.insert(0, ".")

import mlflow
import mlflow.tracking
import pandas as pd
import matplotlib.pyplot as plt

mlflow.set_tracking_uri("sqlite:///mlflow.db")

# EXP005 (qualitative) and EXP007 (precision/recall, not NDCG) excluded from NDCG comparison
NDCG_EXPERIMENTS = ["Tie Strength Decay Function", "Domain Classifier",
                    "FAISS Index Type", "LLM Enrichment Signal Value", "Pathfinding Algorithms"]

client = mlflow.tracking.MlflowClient()
all_runs = {}

for exp_name in NDCG_EXPERIMENTS:
    exp = client.get_experiment_by_name(exp_name)
    if exp is None:
        print(f"[not yet run] {exp_name}")
        continue
    runs = client.search_runs(exp.experiment_id, order_by=["metrics.ndcg_at_10 DESC"])
    all_runs[exp_name] = pd.DataFrame([{
        "variant": r.data.tags.get("mlflow.runName"),
        "ndcg_at_10": r.data.metrics.get("ndcg_at_10"),
        "status": r.data.tags.get("decision", "pending"),
        **r.data.params
    } for r in runs])
    print(f"{exp_name}: {len(runs)} runs")

print(f"\nLoaded {len(all_runs)} experiment(s) with results.")

In [ ]:
# Winners table: best run per experiment
winners_rows = []
for exp_name, df in all_runs.items():
    if df.empty:
        continue
    best = df.iloc[0]
    winners_rows.append({
        "experiment": exp_name,
        "variant": best.get("variant", "—"),
        "ndcg_at_10": best.get("ndcg_at_10", None),
        "status": best.get("status", "pending"),
    })

if winners_rows:
    winners_df = pd.DataFrame(winners_rows)
    print(winners_df.to_string(index=False))

    # Visualization
    valid = winners_df.dropna(subset=["ndcg_at_10"])
    if not valid.empty:
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.barh(valid["experiment"], valid["ndcg_at_10"])
        ax.set_xlabel("Best NDCG@10")
        ax.set_title("Cross-Experiment Winners")
        ax.invert_yaxis()
        plt.tight_layout()
        plt.show()
else:
    print("No experiments have been run yet. Start with EXP001.")

## Production Change Log

Update this cell after each VALIDATED experiment.

| Experiment | Decision | File changed | Function | Date |
|------------|----------|--------------|----------|------|
| (no validated decisions yet) | — | — | — | — |